In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
import torchvision
import torchvision.transforms as transforms
import os
import argparse
from pathlib import Path
import re
import random
import math
import numpy as npM
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Model training, validation, and testing

In [3]:
import os
import random
import numpy as np
import pandas as pd

# -----------------------------
# Config
# -----------------------------
NPZ_PATH = "/content/drive/MyDrive/aim3/saits_masks_1800x2_reintroduced_exact_gaps_capped360_with_bio_numeric.npz"

OUT_DIR  = "/content/drive/MyDrive/aim3/LOCF.MCF"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 1337
N_SPLITS = 5
VAL_WITHIN_TRAIN_P = 0.10

D_IN    = 2
SEQ_LEN = 1080


# ============================================================
# Seed
# ============================================================
def seed_all(seed=1337):
    random.seed(seed)
    np.random.seed(seed)

seed_all(SEED)


# ============================================================
# Helpers
# ============================================================
def rmse_np(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae_np(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean(np.abs(y_true - y_pred)))


def compute_gap_lengths_from_mask(mask_1d, threshold=0.5):
    """
    For each time point in a masked run, assign the full contiguous run length.
    Non-masked points get 0.

    Example:
      mask = [0,0,1,1,1,0,1,1,0]
      out  = [0,0,3,3,3,0,2,2,0]
    """
    mask_bin = (np.asarray(mask_1d) > threshold).astype(np.int32)
    T = len(mask_bin)
    gap_len = np.zeros(T, dtype=np.int32)

    start = 0
    while start < T:
        if mask_bin[start] == 0:
            start += 1
            continue

        end = start
        while end + 1 < T and mask_bin[end + 1] == 1:
            end += 1

        run_len = end - start + 1
        gap_len[start:end + 1] = run_len
        start = end + 1

    return gap_len


def assign_gap_region(g):
    """Exact gap bins for 5-second sampling, measured in samples."""
    g = int(g)
    if 1 <= g < 12: return '<1 min'
    if 12 <= g < 120: return '[1,10) min'
    if 120 <= g < 240: return '[10,20) min'
    if 240 <= g <= 360: return '[20,30] min'
    return None


EXPECTED_BINS = [('<1 min', 1, 11), ('[1,10) min', 12, 119),
                 ('[10,20) min', 120, 239), ('[20,30] min', 240, 360)]
for label, lo, hi in EXPECTED_BINS:
    assert all(assign_gap_region(g) == label for g in range(lo, hi + 1))
assert assign_gap_region(0) is None and assign_gap_region(361) is None
print('Verified all 1–360 gap-length boundaries.')


def make_patient_folds(unique_patients, n_splits=5, seed=1337):
    """
    Robust patient-level folds using np.array_split.
    Works even if number of patients is not divisible by n_splits.
    """
    pats = np.array(unique_patients)
    rng = np.random.default_rng(seed)
    rng.shuffle(pats)
    folds = np.array_split(pats, n_splits)
    return [np.array(f, dtype=pats.dtype) for f in folds]


def split_train_val_patients(trainval_patients, val_fraction=0.10, seed=1337):
    trainval_patients = np.array(trainval_patients)
    n_trainval = len(trainval_patients)

    n_val = max(1, int(round(n_trainval * val_fraction)))
    if n_val >= n_trainval:
        n_val = n_trainval - 1

    rng = np.random.default_rng(seed)
    shuffled = trainval_patients.copy()
    rng.shuffle(shuffled)

    val_pats = shuffled[:n_val]
    train_pats = shuffled[n_val:]
    return np.array(train_pats), np.array(val_pats)


def patient_ids_to_segment_indices(patient_ids_all, selected_patients):
    selected_patients = set(map(int, selected_patients))
    idx = np.array([i for i, p in enumerate(patient_ids_all) if int(p) in selected_patients], dtype=np.int64)
    return idx

def locf_1d(x_in_1d, m_hat_1d, fill_if_no_prev=0.0):
    """
    Last Observation Carried Forward (LOCF)
    - Missing at time t gets last observed value before t.
    - If no previous observed value exists, use fill_if_no_prev (default 0.0).
    """
    x = np.asarray(x_in_1d, dtype=np.float32).copy()
    m = (np.asarray(m_hat_1d) > 0.5).astype(np.int32)
    x = np.nan_to_num(x, nan=0.0)

    T = len(x)
    last = None
    out = np.zeros(T, dtype=np.float32)

    for t in range(T):
        if m[t] == 1:
            last = float(x[t])
            out[t] = float(x[t])
        else:
            out[t] = float(last) if last is not None else float(fill_if_no_prev)

    return out


def mean_carry_forward_1d(x_in_1d, m_hat_1d, fill_if_no_obs=0.0):
    """
    Mean Carried Forward (MCF)
    - Compute mean over OBSERVED values in the segment.
    - Fill every missing point with that mean.
    - If no observed values exist, fill with fill_if_no_obs (default 0.0).
    """
    x = np.asarray(x_in_1d, dtype=np.float32).copy()
    m = (np.asarray(m_hat_1d) > 0.5).astype(np.int32)
    x = np.nan_to_num(x, nan=0.0)

    obs = x[m == 1]
    if obs.size == 0:
        mean_val = float(fill_if_no_obs)
    else:
        mean_val = float(obs.mean())

    out = x.copy()
    out[m == 0] = mean_val
    return out

def weighted_linear_interpolation(left_val, right_val, dist_left, dist_right):
    denom = dist_left + dist_right
    if denom <= 0:
        return 0.5 * (left_val + right_val)
    return (dist_right * left_val + dist_left * right_val) / denom


def weighted_interpolate_1d(x_in_1d, m_hat_1d):
    """
    Fill all missing positions in one 1D sequence using nearest observed
    left/right neighbors (linear weighted interpolation).

    If only one side exists:
      - use nearest observed value from that side
    If no observed value exists at all:
      - fill with 0.0
    """
    x = np.asarray(x_in_1d, dtype=np.float32).copy()
    m = (np.asarray(m_hat_1d) > 0.5).astype(np.int32)

    x = np.nan_to_num(x, nan=0.0)
    T = len(x)

    t = 0
    while t < T:
        if m[t] == 1:
            t += 1
            continue

        start = t
        while t + 1 < T and m[t + 1] == 0:
            t += 1
        end = t

        left_idx = start - 1 if start - 1 >= 0 and m[start - 1] == 1 else None
        right_idx = end + 1 if end + 1 < T and m[end + 1] == 1 else None

        if left_idx is not None and right_idx is not None:
            left_val = float(x[left_idx])
            right_val = float(x[right_idx])

            for k in range(start, end + 1):
                dist_left = float(k - left_idx)
                dist_right = float(right_idx - k)
                x[k] = weighted_linear_interpolation(
                    left_val=left_val,
                    right_val=right_val,
                    dist_left=dist_left,
                    dist_right=dist_right,
                )

        elif left_idx is not None:
            x[start:end + 1] = float(x[left_idx])

        elif right_idx is not None:
            x[start:end + 1] = float(x[right_idx])

        else:
            x[start:end + 1] = 0.0

        t += 1

    return x


def impute_sample(x_in_sample, m_hat_sample, method="WeightedInterp"):
    """
    x_in_sample  : (T,D)
    m_hat_sample : (T,D)
    returns x_completed : (T,D)

    method: "WeightedInterp" | "LOCF" | "MCF"
    """
    T, D = x_in_sample.shape
    out = np.zeros((T, D), dtype=np.float32)

    for d in range(D):
        if method == "WeightedInterp":
            out[:, d] = weighted_interpolate_1d(x_in_sample[:, d], m_hat_sample[:, d])
        elif method == "LOCF":
            out[:, d] = locf_1d(x_in_sample[:, d], m_hat_sample[:, d], fill_if_no_prev=0.0)
        elif method == "MCF":
            out[:, d] = mean_carry_forward_1d(x_in_sample[:, d], m_hat_sample[:, d], fill_if_no_obs=0.0)
        else:
            raise ValueError(f"Unknown method: {method}")

    return out


def collect_predictions_for_indices(
    fold,
    model_name,
    seg_indices,
    X_in_all,
    M_hat_all,
    X_true_all,
    I_mask_all,
    patient_ids_all,
    filenames_all,
    target_names,
):
    """
    Collect pointwise predictions ONLY on artificially masked points (I_mask == 1).
    Prediction is from weighted interpolation completed output.
    """
    rows = []

    for seg_global in seg_indices:
        x_in_s   = X_in_all[seg_global]      # (T,D)
        m_hat_s  = M_hat_all[seg_global]     # (T,D)
        x_true_s = X_true_all[seg_global]    # (T,D)
        i_mask_s = I_mask_all[seg_global]    # (T,D)

        x_completed = impute_sample(x_in_s, m_hat_s, method=model_name)

        for d in range(x_completed.shape[1]):
            gap_len_arr = compute_gap_lengths_from_mask(i_mask_s[:, d])
            masked_t = np.where(i_mask_s[:, d] > 0.5)[0]

            for t in masked_t:
                g = int(gap_len_arr[t])
                rows.append({
                    "fold": fold,
                    "model": model_name,
                    "segment_idx": int(seg_global),
                    "patient_id": int(patient_ids_all[seg_global]),
                    "filename": str(filenames_all[seg_global]),
                    "time_idx": int(t),
                    "target": str(target_names[d]),
                    "y_true": float(x_true_s[t, d]),
                    "y_pred": float(x_completed[t, d]),
                    "abs_err": float(abs(x_true_s[t, d] - x_completed[t, d])),
                    "gap_len": g,
                    "gap_region": assign_gap_region(g),
                })

    return pd.DataFrame(rows)


def metrics_by_target(pred_df_one_split):
    rows = []
    if len(pred_df_one_split) == 0:
        return pd.DataFrame(columns=["target", "mae", "rmse", "n_points"])

    for target, sdf in pred_df_one_split.groupby("target"):
        rows.append({
            "target": target,
            "mae": mae_np(sdf["y_true"], sdf["y_pred"]),
            "rmse": rmse_np(sdf["y_true"], sdf["y_pred"]),
            "n_points": len(sdf),
        })

    return pd.DataFrame(rows)


# ============================================================
# Load NPZ
# ============================================================
data = np.load(NPZ_PATH, allow_pickle=True)

X_in   = data["X_in"]        # (N,T,D)
M_hat  = data["M_hat"]       # (N,T,D)
X_true = data["X"]           # (N,T,D)
I_mask = data["I"]           # (N,T,D)
seg_id = data["segment_ids"] # (N,)

if "filenames" in data.files:
    filenames = np.array([str(x) for x in data["filenames"].tolist()])
else:
    filenames = np.array([f"segment_{i}" for i in range(len(seg_id))])

N, T, D = X_in.shape

if "target_columns" in data.files and len(data["target_columns"]) == D:
    target_names = [str(x) for x in data["target_columns"].tolist()]
elif "columns_all" in data.files and len(data["columns_all"]) == D:
    target_names = [str(x) for x in data["columns_all"].tolist()]
else:
    target_names = [f"channel_{d}" for d in range(D)]

print("Shapes:")
print("  X_in   :", X_in.shape)
print("  M_hat  :", M_hat.shape)
print("  X_true :", X_true.shape)
print("  I_mask :", I_mask.shape)
print("  seg_id :", seg_id.shape)
print("Target names:", target_names)
print("Unique patients:", len(np.unique(seg_id)))

assert D == D_IN, f"Expected D={D_IN}, got D={D}"
assert T == SEQ_LEN, f"Expected T={SEQ_LEN}, got T={T}"


# ============================================================
# Build patient folds
# ============================================================
patients = np.unique(seg_id)
patient_folds = make_patient_folds(patients, n_splits=N_SPLITS, seed=SEED)

print("\nPatient-level folds:")
for i, pats in enumerate(patient_folds, start=1):
    print(f"  Fold {i}: n_test_patients={len(pats)} | patients={sorted(map(int, pats.tolist()))}")


# ============================================================
# CV containers
# ============================================================
all_fold_rows = []
all_pred_rows = []
fold_patient_rows = []


# ============================================================
# 5-fold CV
# ============================================================
for fold, test_pats in enumerate(patient_folds, start=1):
    print("\n" + "#" * 120)
    print(f"FOLD {fold}/{N_SPLITS}")
    print("#" * 120)

    test_pats = np.array(test_pats)
    trainval_pats = np.array([p for p in patients if p not in set(test_pats)])

    train_pats, val_pats = split_train_val_patients(
        trainval_patients=trainval_pats,
        val_fraction=VAL_WITHIN_TRAIN_P,
        seed=SEED + fold
    )

    train_idx = patient_ids_to_segment_indices(seg_id, train_pats)
    val_idx   = patient_ids_to_segment_indices(seg_id, val_pats)
    test_idx  = patient_ids_to_segment_indices(seg_id, test_pats)

    print(f"Patients: train={len(train_pats)}, val={len(val_pats)}, test={len(test_pats)}")
    print(f"Segments: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")

    fold_patient_rows.append({
        "fold": fold,
        "train_patients": ",".join(map(str, sorted(map(int, train_pats.tolist())))),
        "val_patients": ",".join(map(str, sorted(map(int, val_pats.tolist())))),
        "test_patients": ",".join(map(str, sorted(map(int, test_pats.tolist())))),
        "n_train_patients": len(train_pats),
        "n_val_patients": len(val_pats),
        "n_test_patients": len(test_pats),
        "n_train_segments": len(train_idx),
        "n_val_segments": len(val_idx),
        "n_test_segments": len(test_idx),
    })

    # --------------------------------------------------------
    # Test predictions
    # --------------------------------------------------------
    for baseline_name in ["LOCF", "MCF"]:   # add "WeightedInterp" if you want
    # --- VAL ---
      pred_df_val = collect_predictions_for_indices(
          fold=fold,
          model_name=baseline_name,
          seg_indices=val_idx,
          X_in_all=X_in,
          M_hat_all=M_hat,
          X_true_all=X_true,
          I_mask_all=I_mask,
          patient_ids_all=seg_id,
          filenames_all=filenames,
          target_names=target_names,
      )

      # --- TEST ---
      pred_df_test = collect_predictions_for_indices(
          fold=fold,
          model_name=baseline_name,
          seg_indices=test_idx,
          X_in_all=X_in,
          M_hat_all=M_hat,
          X_true_all=X_true,
          I_mask_all=I_mask,
          patient_ids_all=seg_id,
          filenames_all=filenames,
          target_names=target_names,
      )

      all_pred_rows.append(pred_df_test)

      val_metrics_df = metrics_by_target(pred_df_val)
      test_metrics_df = metrics_by_target(pred_df_test)

      merged = pd.merge(
          val_metrics_df.rename(columns={"mae": "val_mae", "rmse": "val_rmse", "n_points": "n_val_points"}),
          test_metrics_df.rename(columns={"mae": "test_mae", "rmse": "test_rmse", "n_points": "n_test_points"}),
          on="target",
          how="outer"
      )

      for _, row in merged.iterrows():
          target = row["target"]

          print(
              f"[Fold {fold}] [{baseline_name}] {target} | "
              f"VAL MAE={row['val_mae']:.5f}, RMSE={row['val_rmse']:.5f} | "
              f"TEST MAE={row['test_mae']:.5f}, RMSE={row['test_rmse']:.5f}"
          )

          all_fold_rows.append({
              "fold": fold,
              "target": target,
              "model": baseline_name,
              "val_mae": row["val_mae"],
              "val_rmse": row["val_rmse"],
              "test_mae": row["test_mae"],
              "test_rmse": row["test_rmse"],
              "n_val_points": row["n_val_points"],
              "n_test_points": row["n_test_points"],
              "n_train_patients": len(train_pats),
              "n_val_patients": len(val_pats),
              "n_test_patients": len(test_pats),
              "n_train_segments": len(train_idx),
              "n_val_segments": len(val_idx),
              "n_test_segments": len(test_idx),
          })

# ============================================================
# Combine results
# ============================================================
fold_df = pd.DataFrame(all_fold_rows)
pred_df = pd.concat(all_pred_rows, axis=0, ignore_index=True)
fold_patient_df = pd.DataFrame(fold_patient_rows)

fold_csv = os.path.join(OUT_DIR, "cv5_fold_results.csv")
pred_csv = os.path.join(OUT_DIR, "cv5_test_predictions_pooled.csv")
split_csv = os.path.join(OUT_DIR, "cv5_patient_split_summary.csv")

fold_df.to_csv(fold_csv, index=False)
pred_df.to_csv(pred_csv, index=False)
fold_patient_df.to_csv(split_csv, index=False)


# ============================================================
# Summary: mean ± std across folds
# ============================================================
summary_df = (
    fold_df
    .groupby(["target", "model"], as_index=False)
    .agg(
        val_mae_mean=("val_mae", "mean"),
        val_mae_std=("val_mae", "std"),
        val_rmse_mean=("val_rmse", "mean"),
        val_rmse_std=("val_rmse", "std"),
        test_mae_mean=("test_mae", "mean"),
        test_mae_std=("test_mae", "std"),
        test_rmse_mean=("test_rmse", "mean"),
        test_rmse_std=("test_rmse", "std"),
        n_folds=("fold", "count"),
        total_test_points=("n_test_points", "sum"),
    )
)

summary_csv = os.path.join(OUT_DIR, "cv5_summary_mean_std.csv")
summary_df.to_csv(summary_csv, index=False)


# ============================================================
# Pooled out-of-fold test metrics
# ============================================================
pooled_rows = []
for (target, model), sdf in pred_df.groupby(["target", "model"]):
    pooled_rows.append({
        "target": target,
        "model": model,
        "pooled_test_mae": mae_np(sdf["y_true"], sdf["y_pred"]),
        "pooled_test_rmse": rmse_np(sdf["y_true"], sdf["y_pred"]),
        "n_points": len(sdf),
    })

pooled_df = pd.DataFrame(pooled_rows)
pooled_csv = os.path.join(OUT_DIR, "cv5_pooled_test_metrics.csv")
pooled_df.to_csv(pooled_csv, index=False)


# ============================================================
# Gap-region summaries
# ============================================================
pred_df_gap = pred_df[pred_df["gap_region"].notna()].copy()

pred_gap_csv = os.path.join(OUT_DIR, "cv5_test_predictions_pooled_with_gap_region.csv")
pred_df_gap.to_csv(pred_gap_csv, index=False)

# pooled by gap region
pooled_gap_rows = []
for (target, model, gap_region), sdf in pred_df_gap.groupby(["target", "model", "gap_region"]):
    pooled_gap_rows.append({
        "target": target,
        "model": model,
        "gap_region": str(gap_region),
        "pooled_test_mae": mae_np(sdf["y_true"], sdf["y_pred"]),
        "pooled_test_rmse": rmse_np(sdf["y_true"], sdf["y_pred"]),
        "n_points": len(sdf),
    })

pooled_gap_df = pd.DataFrame(pooled_gap_rows)
pooled_gap_csv = os.path.join(OUT_DIR, "cv5_pooled_test_metrics_by_gap_region.csv")
pooled_gap_df.to_csv(pooled_gap_csv, index=False)

# fold-wise by gap region
fold_gap_rows = []
for (fold, target, model, gap_region), sdf in pred_df_gap.groupby(["fold", "target", "model", "gap_region"]):
    fold_gap_rows.append({
        "fold": fold,
        "target": target,
        "model": model,
        "gap_region": str(gap_region),
        "test_mae": mae_np(sdf["y_true"], sdf["y_pred"]),
        "test_rmse": rmse_np(sdf["y_true"], sdf["y_pred"]),
        "n_points": len(sdf),
    })

fold_gap_df = pd.DataFrame(fold_gap_rows)
fold_gap_csv = os.path.join(OUT_DIR, "cv5_fold_metrics_by_gap_region.csv")
fold_gap_df.to_csv(fold_gap_csv, index=False)

gap_summary_df = (
    fold_gap_df
    .groupby(["target", "model", "gap_region"], as_index=False)
    .agg(
        test_mae_mean=("test_mae", "mean"),
        test_mae_std=("test_mae", "std"),
        test_rmse_mean=("test_rmse", "mean"),
        test_rmse_std=("test_rmse", "std"),
        n_folds=("fold", "count"),
        total_points=("n_points", "sum"),
    )
)

gap_summary_csv = os.path.join(OUT_DIR, "cv5_summary_mean_std_by_gap_region.csv")
gap_summary_df.to_csv(gap_summary_csv, index=False)


# ============================================================
# Print final paths
# ============================================================
print("\nSaved:")
print("  fold results     :", fold_csv)
print("  pooled preds     :", pred_csv)
print("  pooled preds+gap :", pred_gap_csv)
print("  mean±std         :", summary_csv)
print("  pooled test      :", pooled_csv)
print("  patient splits   :", split_csv)
print("  pooled gap test  :", pooled_gap_csv)
print("  fold gap metrics :", fold_gap_csv)
print("  gap mean±std     :", gap_summary_csv)

print("\nPer-fold results:")
print(fold_df)

print("\nMean ± std summary:")
print(summary_df)

print("\nPooled out-of-fold test metrics:")
print(pooled_df)

print("\nPooled out-of-fold test metrics by gap region:")
print(pooled_gap_df)

print("\nMean ± std by gap region:")
print(gap_summary_df)

print("\nPatient split summary:")
print(fold_patient_df)

# ============================================================
# Publication-style five-fold MAE/MAPE report (same as CSDI)
# ============================================================
EPS = 1e-8
REGIONS = ['<1 min', '[1,10) min', '[10,20) min', '[20,30] min', 'ALL']

pred_df['gap_region'] = pred_df['gap_len'].map(assign_gap_region)
print('\nArtificially masked POINT counts by region:')
print(pred_df['gap_region'].fillna('OUT_OF_RANGE').value_counts().to_string())
print('Max gap length:', pred_df['gap_len'].max())
if pred_df['gap_region'].isna().any():
    print('WARNING: Gap lengths outside 1–360 exist; these contribute to ALL only.')

valid_df = pred_df[np.isfinite(pred_df['y_true']) & np.isfinite(pred_df['y_pred'])].copy()


def calculate_metrics(frame):
    yt = frame['y_true'].to_numpy(dtype=np.float64)
    yp = frame['y_pred'].to_numpy(dtype=np.float64)
    error = np.abs(yt - yp)
    valid_mape = np.abs(yt) > EPS
    return pd.Series({
        'MAE': float(error.mean()) if len(error) else np.nan,
        'MAPE_pct': (float(100 * np.mean(error[valid_mape] / np.abs(yt[valid_mape])))
                     if valid_mape.any() else np.nan),
        'n_points': len(yt),
        'n_mape_points': int(valid_mape.sum())
    })

by_gap = valid_df[valid_df['gap_region'].notna()].groupby(
    ['fold', 'target', 'model', 'gap_region'], observed=True
)[['y_true', 'y_pred']].apply(calculate_metrics).reset_index()

all_regions = valid_df.groupby(
    ['fold', 'target', 'model'], observed=True
)[['y_true', 'y_pred']].apply(calculate_metrics).reset_index()
all_regions['gap_region'] = 'ALL'
foldwise = pd.concat([by_gap, all_regions], ignore_index=True)
foldwise['gap_region'] = pd.Categorical(foldwise['gap_region'], categories=REGIONS, ordered=True)
foldwise = foldwise.sort_values(['target', 'model', 'gap_region', 'fold']).reset_index(drop=True)

summary = foldwise.groupby(['target', 'model', 'gap_region'], observed=True).agg(
    MAE_mean=('MAE', 'mean'), MAE_std=('MAE', 'std'),
    MAPE_mean_pct=('MAPE_pct', 'mean'), MAPE_std_pct=('MAPE_pct', 'std'),
    n_folds_MAE=('MAE', 'count'), n_folds_MAPE=('MAPE_pct', 'count'),
    total_points=('n_points', 'sum'), total_mape_points=('n_mape_points', 'sum')
).reset_index()

def fmt(mean, std):
    if pd.isna(mean): return 'NA'
    return f'{mean:.2f} ± {std:.2f}' if pd.notna(std) else f'{mean:.2f} ± NA'

formatted = summary.copy()
formatted['MAE'] = [fmt(m, s) for m, s in zip(formatted.MAE_mean, formatted.MAE_std)]
formatted['MAPE(%)'] = [fmt(m, s) for m, s in zip(formatted.MAPE_mean_pct, formatted.MAPE_std_pct)]
paper = formatted.pivot(index=['target', 'model'], columns='gap_region', values=['MAE', 'MAPE(%)'])
paper = paper.swaplevel(0, 1, axis=1)
paper = paper.reindex(columns=pd.MultiIndex.from_product([REGIONS, ['MAE', 'MAPE(%)']]))
paper.columns = [f'{gap} | {metric}' for gap, metric in paper.columns]
paper = paper.reset_index()

pooled_gap = valid_df[valid_df['gap_region'].notna()].groupby(
    ['target', 'model', 'gap_region'], observed=True
)[['y_true', 'y_pred']].apply(calculate_metrics).reset_index()
pooled_all = valid_df.groupby(['target', 'model'], observed=True)[
    ['y_true', 'y_pred']].apply(calculate_metrics).reset_index()
pooled_all['gap_region'] = 'ALL'
pooled = pd.concat([pooled_gap, pooled_all], ignore_index=True)

for filename, frame in {
    'locf_mcf_paper4_foldwise.csv': foldwise,
    'locf_mcf_paper4_mean_std.csv': summary,
    'locf_mcf_paper4_pooled.csv': pooled,
    'locf_mcf_paper4_paper_table.csv': paper,
}.items():
    dest = os.path.join(OUT_DIR, filename)
    frame.to_csv(dest, index=False)
    print('Saved:', dest)

print('\nLOCF / MCF FIVE-FOLD MEAN ± STD:')
print(paper.to_string(index=False))
print('\nFold coverage per model, target, region:')
print(summary[['target','model','gap_region','n_folds_MAE','n_folds_MAPE']].to_string(index=False))


Verified all 1–360 gap-length boundaries.
Shapes:
  X_in   : (3817, 1080, 2)
  M_hat  : (3817, 1080, 2)
  X_true : (3817, 1080, 2)
  I_mask : (3817, 1080, 2)
  seg_id : (3817,)
Target names: ['channel_0', 'channel_1']
Unique patients: 64

Patient-level folds:
  Fold 1: n_test_patients=13 | patients=[6, 13, 16, 22, 30, 31, 47, 52, 54, 59, 65, 68, 71]
  Fold 2: n_test_patients=13 | patients=[3, 9, 10, 11, 12, 15, 27, 33, 34, 36, 61, 67, 69]
  Fold 3: n_test_patients=13 | patients=[4, 19, 20, 24, 28, 38, 43, 44, 50, 56, 62, 64, 66]
  Fold 4: n_test_patients=13 | patients=[7, 8, 14, 17, 21, 25, 32, 45, 49, 51, 58, 63, 70]
  Fold 5: n_test_patients=12 | patients=[1, 2, 5, 23, 26, 35, 37, 48, 53, 55, 57, 60]

########################################################################################################################
FOLD 1/5
########################################################################################################################
Patients: train=46, val=5, test=13
S